# Análisis estadístico de señales  
## Práctica 3 - PARTE 2 (SEÑAL EEG)

**Integrantes:**  
- Sara Mejía Zapata  
- María Camila Tobón Úsuga

**Asignatura:** Bioseñales y Sistemas


---

In [18]:
import mne
import pandas as pd 
import numpy as np

In [14]:
import mne
import os

base_path = 'Lab_EEG_Imagineria'

sujeto = 'sub_1'
ruta_sujeto = os.path.join(base_path, sujeto)

# escoger un archivo .set (por ejemplo el primero)
archivos = os.listdir(ruta_sujeto)

for archivo in archivos:
    if archivo.endswith('.set'):
        ruta_archivo = os.path.join(ruta_sujeto, archivo)
        break

# 🔥 AQUÍ ESTÁ EL CAMBIO IMPORTANTE
raw = mne.io.read_raw_eeglab(ruta_archivo, preload=True)

# eventos
events, event_id = mne.events_from_annotations(raw)

print("Eventos Encontrados :", event_id)

Used Annotations descriptions: [np.str_('TASK2T0'), np.str_('TASK2T1'), np.str_('TASK2T2')]
Eventos Encontrados : {np.str_('TASK2T0'): 1, np.str_('TASK2T1'): 2, np.str_('TASK2T2'): 3}


In [15]:
# Función RMS
def calcular_rms(epocas):
    rms_epoca = np.sqrt(np.mean(epocas**2, axis=2))
    rms_prom = np.mean(rms_epoca, axis=0)
    return rms_prom

In [16]:
# Listas para almacenar resultados
rms_T1_list = []
rms_T2_list = []
canales = None

# Recorrer sujetos
for sujeto in os.listdir(base_path):
    
    ruta_sujeto = os.path.join(base_path, sujeto)
    
    if os.path.isdir(ruta_sujeto):
        
        # Buscar archivos .set dentro del sujeto
        for archivo in os.listdir(ruta_sujeto):
            
            if archivo.endswith('.set') and 'run-4' in archivo:
                
                ruta_archivo = os.path.join(ruta_sujeto, archivo)
                
                # Cargar señal
                raw = mne.io.read_raw_eeglab(ruta_archivo, preload=True)
                
                # Eventos
                events, event_id = mne.events_from_annotations(raw)
                
                # Crear épocas
                epocas = mne.Epochs(raw, events, event_id, preload=True)
                
                # Separar clases
                epocas_T1 = epocas['TASK2T1']
                epocas_T2 = epocas['TASK2T2']
                
                # Guardar nombres de canales (solo una vez)
                if canales is None:
                    canales = epocas.ch_names
                
                # Calcular RMS
                rms_T1 = calcular_rms(epocas_T1.get_data())
                rms_T2 = calcular_rms(epocas_T2.get_data())
                
                # Guardar resultados
                rms_T1_list.append(rms_T1)
                rms_T2_list.append(rms_T2)

Used Annotations descriptions: [np.str_('TASK2T0'), np.str_('TASK2T1'), np.str_('TASK2T2')]
Not setting metadata
30 matching events found
Setting baseline interval to [-0.2, 0.0] s
Applying baseline correction (mode: mean)
0 projection items activated
Using data from preloaded Raw for 30 events and 113 original time points ...
1 bad epochs dropped
Used Annotations descriptions: [np.str_('TASK2T0'), np.str_('TASK2T1'), np.str_('TASK2T2')]
Not setting metadata
30 matching events found
Setting baseline interval to [-0.2, 0.0] s
Applying baseline correction (mode: mean)
0 projection items activated
Using data from preloaded Raw for 30 events and 113 original time points ...
1 bad epochs dropped
Used Annotations descriptions: [np.str_('TASK2T0'), np.str_('TASK2T1'), np.str_('TASK2T2')]
Not setting metadata
30 matching events found
Setting baseline interval to [-0.2, 0.0] s
Applying baseline correction (mode: mean)
0 projection items activated
Using data from preloaded Raw for 30 events and 

In [17]:
df_T1 = pd.DataFrame(rms_T1_list, columns=canales)
df_T2 = pd.DataFrame(rms_T2_list, columns=canales)

print(df_T1)
print(df_T2)

        Fc5       Fc3       Fc1       Fcz       Fc2       Fc4       Fc6  \
0  0.000048  0.000050  0.000052  0.000050  0.000050  0.000045  0.000036   
1  0.000063  0.000051  0.000060  0.000059  0.000050  0.000044  0.000069   
2  0.000022  0.000022  0.000022  0.000023  0.000021  0.000019  0.000016   
3  0.000073  0.000069  0.000061  0.000059  0.000059  0.000046  0.000045   
4  0.000029  0.000028  0.000029  0.000022  0.000022  0.000021  0.000021   
5  0.000069  0.000032  0.000048  0.000045  0.000037  0.000048  0.000047   
6  0.000025  0.000023  0.000023  0.000022  0.000021  0.000021  0.000020   
7  0.000041  0.000040  0.000040  0.000039  0.000039  0.000039  0.000037   
8  0.000038  0.000032  0.000033  0.000033  0.000033  0.000033  0.000037   
9  0.000099  0.000070  0.000063  0.000062  0.000065  0.000065  0.000109   

         C5        C3        C1  ...        P8       Po7       Po3       Poz  \
0  0.000048  0.000052  0.000054  ...  0.000041  0.000057  0.000058  0.000057   
1  0.000049  0

In [19]:
print(df_T1.shape)
print(df_T2.shape)

(10, 64)
(10, 64)
